In [ ]:
import numpy as np
import pandas as pd
import neurokit2 as nk
from pathlib import Path
from tqdm import tqdm
from sklearn.ensemble import RandomForestClassifier
from sklearn.multioutput import MultiOutputClassifier
from sklearn.metrics import f1_score
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import shap

In [ ]:
PROCESSED_DIR = Path(r"C:\ECG_Project\training\processed_signals")
STAGE4_CSV_PATH = r"C:\ECG_Project\training\All excel final\ecg_metadata_stage4_final.csv"
OUTPUT_CSV_PATH = r"C:\ECG_Project\training\All excel files\ecg_fiducial_features.csv"
FAILED_LOG_PATH = r"C:\ECG_Project\training\All excel files\stage5_failed_records.csv"
metadata_df = pd.read_csv(r"C:\ECG_Project\training\All excel files\ecg_metadata_train_test_split.csv")
FEDERATED_CLIENTS = ["chapman_shaoxing", "cpsc_2018", "georgia", "ningbo", "ptb-xl"]
FINAL_CLASSES = ["AF", "IAVB", "LAD", "LBBB", "NSIVCB", "NSR", "PAC", "QAb", "RBBB", "SB", "STach", "TAb"]

In [1]:
SAMPLING_RATE = 500        # matches Stage 4's TARGET_FS
LEAD_INDEX = 1              # Lead II — standard for P/QRS/T delineation
LEAD_NAME = "II"

# Checkpoint the running results every N records, in case of interruption
CHECKPOINT_INTERVAL = 5000

def extract_fiducial_features(record_id: str, processed_dir: Path,
                               sampling_rate: int = SAMPLING_RATE,
                               lead_index: int = LEAD_INDEX) -> dict | None:
    """
    Runs neurokit2's ECG pipeline on ONE lead of a preprocessed signal,
    and pulls out the fiducial features needed for the SHAP layer:
    P/QRS/T amplitudes + PR/QT/RR intervals.
    Returns None if the signal is too noisy/short for reliable delineation
    (neurokit2 will throw on these — that's expected for a small fraction).
    """
    try:
        signal = np.load(processed_dir / f"{record_id}.npy")
        lead_signal = signal[lead_index, :]   # shape (5000,) — single lead, already filtered+normalized

        # neurokit2's full pipeline: clean -> detect R-peaks -> delineate P/QRS/T
        signals_df, info = nk.ecg_process(lead_signal, sampling_rate=sampling_rate)

        # --- R-peaks -> RR intervals (in seconds) ---
        r_peaks = info["ECG_R_Peaks"]
        if len(r_peaks) < 2:
            return None   # can't compute RR interval with fewer than 2 beats detected
        rr_intervals = np.diff(r_peaks) / sampling_rate
        mean_rr = float(np.mean(rr_intervals))

        # --- Wave boundaries/peaks located by nk.ecg_delineate (already run inside ecg_process) ---
        p_peaks = np.array(info["ECG_P_Peaks"], dtype=float)
        r_onsets = np.array(info["ECG_R_Onsets"], dtype=float)
        q_peaks = np.array(info["ECG_Q_Peaks"], dtype=float)
        s_peaks = np.array(info["ECG_S_Peaks"], dtype=float)
        t_peaks = np.array(info["ECG_T_Peaks"], dtype=float)
        t_offsets = np.array(info["ECG_T_Offsets"], dtype=float)
        p_onsets = np.array(info["ECG_P_Onsets"], dtype=float)

        def safe_amplitude(peak_indices):
            """Average signal amplitude at a set of wave-peak sample indices, ignoring NaNs (undetected beats)."""
            valid = peak_indices[~np.isnan(peak_indices)].astype(int)
            valid = valid[(valid >= 0) & (valid < len(lead_signal))]
            if len(valid) == 0:
                return np.nan
            return float(np.mean(lead_signal[valid]))

        def safe_interval(start_indices, end_indices, sampling_rate):
            """Mean time (seconds) between matched start/end index pairs, ignoring incomplete beats."""
            n = min(len(start_indices), len(end_indices))
            diffs = []
            for i in range(n):
                if not (np.isnan(start_indices[i]) or np.isnan(end_indices[i])):
                    diffs.append((end_indices[i] - start_indices[i]) / sampling_rate)
            return float(np.mean(diffs)) if diffs else np.nan

        features = {
            "record_id": record_id,
            "lead_used": LEAD_NAME,
            "mean_rr_interval": mean_rr,
            "heart_rate_bpm": 60.0 / mean_rr if mean_rr > 0 else np.nan,
            "p_wave_amplitude": safe_amplitude(p_peaks),
            "qrs_amplitude": safe_amplitude(q_peaks),   # Q as QRS-complex reference point
            "t_wave_amplitude": safe_amplitude(t_peaks),
            "pr_interval": safe_interval(p_onsets, r_onsets, sampling_rate),
            "qt_interval": safe_interval(q_peaks, t_offsets, sampling_rate),
            "qrs_duration": safe_interval(q_peaks, s_peaks, sampling_rate),
            "n_beats_detected": len(r_peaks),
        }
        return features

    except Exception as e:
        print(f"  [FAILED] {record_id}: {e}")
        return None


def run_fiducial_extraction(metadata_df: pd.DataFrame):
    all_features = []
    failed_records = []

    # RESUME LOGIC: skip records already present in a previous partial run
    already_done = set()
    if Path(OUTPUT_CSV_PATH).exists():
        existing_df = pd.read_csv(OUTPUT_CSV_PATH)
        already_done = set(existing_df["record_id"])
        all_features = existing_df.to_dict("records")
        print(f"Resuming — {len(already_done)} records already processed")

    record_ids = metadata_df["record_id"].tolist()

    for idx, record_id in enumerate(tqdm(record_ids, desc="Extracting fiducial features")):
        if record_id in already_done:
            continue

        features = extract_fiducial_features(record_id, PROCESSED_DIR)
        if features is not None:
            all_features.append(features)
        else:
            failed_records.append(record_id)

        # Periodic checkpoint save
        if idx % CHECKPOINT_INTERVAL == 0 and idx > 0:
            pd.DataFrame(all_features).to_csv(OUTPUT_CSV_PATH, index=False)
            pd.DataFrame({"record_id": failed_records}).to_csv(FAILED_LOG_PATH, index=False)

    # Final save
    features_df = pd.DataFrame(all_features)
    features_df.to_csv(OUTPUT_CSV_PATH, index=False)
    pd.DataFrame({"record_id": failed_records}).to_csv(FAILED_LOG_PATH, index=False)

    print(f"\nSuccessfully extracted: {len(features_df)}")
    print(f"Failed (too noisy for reliable delineation): {len(failed_records)}")
    print(f"Saved: {OUTPUT_CSV_PATH}")

    return features_df


if __name__ == "__main__":
    metadata_df = pd.read_csv(STAGE4_CSV_PATH)
    print(f"Total records to process: {len(metadata_df)}")

    features_df = run_fiducial_extraction(metadata_df)
    print(features_df.head())
    print(f"\nFinal shape: {features_df.shape}")

Total records to process: 72509


C:\Users\slalu\AppData\Local\Programs\Python\Python313\Lib\site-packages\neurokit2\signal\signal_period.py:83: NeuroKitWarning: Too few peaks detected to compute the rate. Returning empty vector.
  warn(
C:\Users\slalu\AppData\Local\Programs\Python\Python313\Lib\site-packages\neurokit2\signal\signal_period.py:83: NeuroKitWarning: Too few peaks detected to compute the rate. Returning empty vector.
  warn(
C:\Users\slalu\AppData\Local\Programs\Python\Python313\Lib\site-packages\neurokit2\signal\signal_period.py:83: NeuroKitWarning: Too few peaks detected to compute the rate. Returning empty vector.
  warn(
C:\Users\slalu\AppData\Local\Programs\Python\Python313\Lib\site-packages\neurokit2\signal\signal_period.py:83: NeuroKitWarning: Too few peaks detected to compute the rate. Returning empty vector.
  warn(


  [FAILED] JS10030: integer division or modulo by zero
  [FAILED] JS10032: integer division or modulo by zero
  [FAILED] JS10034: integer division or modulo by zero
  [FAILED] JS10035: integer division or modulo by zero


C:\Users\slalu\AppData\Local\Programs\Python\Python313\Lib\site-packages\neurokit2\signal\signal_period.py:83: NeuroKitWarning: Too few peaks detected to compute the rate. Returning empty vector.
  warn(


  [FAILED] JS10123: integer division or modulo by zero


...

  [FAILED] HR08348: cannot convert float NaN to integer


C:\Users\slalu\AppData\Local\Programs\Python\Python313\Lib\site-packages\neurokit2\signal\signal_period.py:83: NeuroKitWarning: Too few peaks detected to compute the rate. Returning empty vector.
  warn(


  [FAILED] HR08714: cannot convert float NaN to integer


Extracting fiducial features: 100%|████████████████████████████████████████████| 72509/72509 [5:35:17<00:00,  3.60it/s]



Successfully extracted: 72359
Failed (too noisy for reliable delineation): 150
Saved: C:\ECG_Project\training\ecg_fiducial_features.csv
  record_id lead_used  mean_rr_interval  heart_rate_bpm  p_wave_amplitude  \
0   JS00001        II          0.512556      117.060481          0.460115   
1   JS00002        II          1.160571       51.698671          0.320991   
2   JS00004        II          1.125500       53.309640         -0.084264   
3   JS00006        II          1.059750       56.617127         -0.049934   
4   JS00007        II          0.626933       95.703956          0.418069   

   qrs_amplitude  t_wave_amplitude  pr_interval  qt_interval  qrs_duration  \
0      -0.381637          0.260966     0.082000     0.208444      0.100556   
1      -0.735753          0.741837     0.127000     0.409000      0.167000   
2      -0.519227          0.914195     0.069333     0.467250      0.155500   
3      -0.826430          1.875153     0.055250     0.417111      0.134889   
4      -0.

In [ ]:
fiducial_df = pd.read_csv(r"C:\ECG_Project\training\All excel files\ecg_fiducial_features.csv")
FEATURE_COLS = ["mean_rr_interval", "heart_rate_bpm", "p_wave_amplitude",
                 "qrs_amplitude", "t_wave_amplitude", "pr_interval",
                 "qt_interval", "qrs_duration", "n_beats_detected"]

In [4]:
# Bring in source_hospital + split, so imputation can be done per-hospital
# and we keep the same train/test assignment as your ResNet pipeline
fiducial_df = fiducial_df.merge(
    metadata_df[["record_id", "source_hospital", "split"] + FINAL_CLASSES],
    on="record_id", how="inner"
)
print(f"After merge: {fiducial_df.shape}")

print("\nNull counts before imputation:")
print(fiducial_df[FEATURE_COLS].isnull().sum())

# Per-hospital median imputation — fit only on TRAIN rows, applied to both
# train and test (avoids leaking test-set statistics into training, same
# discipline as your normalization/scaling steps elsewhere in the pipeline)
for col in FEATURE_COLS:
    train_medians = fiducial_df[fiducial_df["split"] == "train"].groupby("source_hospital")[col].median()

    def fill_value(row):
        if pd.isna(row[col]):
            return train_medians.get(row["source_hospital"], fiducial_df[col].median())
        return row[col]

    fiducial_df[col] = fiducial_df.apply(fill_value, axis=1)

print("\nNull counts after imputation:")
print(fiducial_df[FEATURE_COLS].isnull().sum())   # should all be 0 now

fiducial_df.to_csv(r"C:\ECG_Project\training\ecg_fiducial_features_clean.csv", index=False)
print(f"\nSaved: ecg_fiducial_features_clean.csv, shape={fiducial_df.shape}")

After merge: (72285, 25)

Null counts before imputation:
mean_rr_interval      0
heart_rate_bpm        0
p_wave_amplitude     51
qrs_amplitude        64
t_wave_amplitude    555
pr_interval         163
qt_interval         715
qrs_duration         70
n_beats_detected      0
dtype: int64

Null counts after imputation:
mean_rr_interval    0
heart_rate_bpm      0
p_wave_amplitude    0
qrs_amplitude       0
t_wave_amplitude    0
pr_interval         0
qt_interval         0
qrs_duration        0
n_beats_detected    0
dtype: int64

Saved: ecg_fiducial_features_clean.csv, shape=(72285, 25)


In [10]:
train_df = fiducial_df[fiducial_df["split"] == "train"]
test_df = fiducial_df[fiducial_df["split"] == "test"]

X_train, y_train = train_df[FEATURE_COLS], train_df[FINAL_CLASSES]
X_test, y_test = test_df[FEATURE_COLS], test_df[FINAL_CLASSES]

# One RandomForest per class (multi-label), same 12-class structure as your ResNet

base_rf = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1)
clf = MultiOutputClassifier(base_rf)
clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)
micro_f1 = f1_score(y_test, y_pred, average="micro", zero_division=0)
macro_f1 = f1_score(y_test, y_pred, average="macro", zero_division=0)
print(f"Fiducial-feature classifier — micro_f1={micro_f1:.4f}  macro_f1={macro_f1:.4f}")

joblib.dump(clf, r"C:\ECG_Project\training\fiducial_rf_classifier.pkl")
print("Saved: fiducial_rf_classifier.pkl")

Fiducial-feature classifier — micro_f1=0.6353  macro_f1=0.2638
Saved: fiducial_rf_classifier.pkl


In [14]:
per_class_f1 = f1_score(y_test, y_pred, average=None, zero_division=0)
for cls, f1 in zip(FINAL_CLASSES, per_class_f1):
    print(f"{cls}: {f1:.4f}")

AF: 0.2487
IAVB: 0.0492
LAD: 0.1811
LBBB: 0.0268
NSIVCB: 0.0000
NSR: 0.7854
PAC: 0.0000
QAb: 0.0000
RBBB: 0.0000
SB: 0.8713
STach: 0.8729
TAb: 0.1302


In [ ]:
clf = joblib.load(r"C:\ECG_Project\training\fiducial_rf_classifier.pkl")
fiducial_df = pd.read_csv(r"C:\ECG_Project\training\All excel files\ecg_fiducial_features_clean.csv")

In [20]:
test_df = fiducial_df[fiducial_df["split"] == "test"]
shap_results = {}

for class_idx, class_name in enumerate(FINAL_CLASSES):
    estimator = clf.estimators_[class_idx]   # the actual TRAINED model for this class
    class_explainer = shap.TreeExplainer(estimator)

    for hospital in FEDERATED_CLIENTS:
        hosp_test = test_df[test_df["source_hospital"] == hospital]
        pos_records = hosp_test[hosp_test[class_name] == 1]

        if len(pos_records) < 5:
            print(f"  [{hospital}/{class_name}] n={len(pos_records)}, below min — skipped")
            continue

        X_subset = pos_records[FEATURE_COLS]
        shap_values = class_explainer.shap_values(X_subset)

        if isinstance(shap_values, list):
            shap_values = shap_values[1]   # positive class

        mean_abs_shap = np.abs(shap_values).mean(axis=0)
        shap_results[f"{hospital}__{class_name}"] = dict(zip(FEATURE_COLS, mean_abs_shap))
        print(f"  [{hospital}/{class_name}] n={len(pos_records)} — SHAP computed")

shap_summary_df = pd.DataFrame(shap_results).T
shap_summary_df.to_csv(r"C:\ECG_Project\training\shap_feature_importance_summary.csv")
print(f"\nSaved: shap_feature_importance_summary.csv, shape={shap_summary_df.shape}")

  [chapman_shaoxing/AF] n=355 — SHAP computed
  [cpsc_2018/AF] n=275 — SHAP computed
  [georgia/AF] n=113 — SHAP computed
  [ningbo/AF] n=0, below min — skipped
  [ptb-xl/AF] n=303 — SHAP computed
  [chapman_shaoxing/IAVB] n=50 — SHAP computed
  [cpsc_2018/IAVB] n=166 — SHAP computed
  [georgia/IAVB] n=154 — SHAP computed
  [ningbo/IAVB] n=175 — SHAP computed
  [ptb-xl/IAVB] n=158 — SHAP computed
  [chapman_shaoxing/LAD] n=75 — SHAP computed
  [cpsc_2018/LAD] n=0, below min — skipped
  [georgia/LAD] n=186 — SHAP computed
  [ningbo/LAD] n=229 — SHAP computed
  [ptb-xl/LAD] n=1025 — SHAP computed
  [chapman_shaoxing/LBBB] n=38 — SHAP computed
  [cpsc_2018/LBBB] n=55 — SHAP computed
  [georgia/LBBB] n=44 — SHAP computed
  [ningbo/LBBB] n=49 — SHAP computed
  [ptb-xl/LBBB] n=106 — SHAP computed
  [chapman_shaoxing/NSIVCB] n=47 — SHAP computed
  [cpsc_2018/NSIVCB] n=0, below min — skipped
  [georgia/NSIVCB] n=41 — SHAP computed
  [ningbo/NSIVCB] n=107 — SHAP computed
  [ptb-xl/NSIVCB] n=157

In [22]:
shap_summary_df = pd.read_csv(r"C:\ECG_Project\training\shap_feature_importance_summary.csv", index_col=0)
all_possible = [f"{h}__{c}" for h in FEDERATED_CLIENTS for c in FINAL_CLASSES]
missing = [k for k in all_possible if k not in shap_summary_df.index]
print(f"Excluded cells ({len(missing)}):")
for m in missing:
    print(f"  {m}")

Excluded cells (6):
  cpsc_2018__LAD
  cpsc_2018__NSIVCB
  cpsc_2018__QAb
  cpsc_2018__TAb
  ningbo__AF
  ptb-xl__RBBB


In [32]:
test_df = fiducial_df[fiducial_df["split"] == "test"]
shap_results = {}

for class_idx, class_name in enumerate(FINAL_CLASSES):
    estimator = clf.estimators_[class_idx]
    class_explainer = shap.TreeExplainer(estimator)

    for hospital in FEDERATED_CLIENTS:
        hosp_test = test_df[test_df["source_hospital"] == hospital]
        pos_records = hosp_test[hosp_test[class_name] == 1]

        if len(pos_records) < 5:
            print(f"  [{hospital}/{class_name}] n={len(pos_records)}, below min — skipped")
            continue

        X_subset = pos_records[FEATURE_COLS]
        shap_values = class_explainer.shap_values(X_subset)

        # Handle BOTH possible SHAP output shapes, depending on library version:
        if isinstance(shap_values, list):
            # Old API: list of 2 arrays, one per class -> take positive class
            sv = shap_values[1]
        elif isinstance(shap_values, np.ndarray) and shap_values.ndim == 3:
            # New API: single 3D array (n_samples, n_features, n_classes) -> take positive class
            sv = shap_values[:, :, 1]
        else:
            # Already 2D (n_samples, n_features) -- some versions return this directly
            sv = shap_values

        mean_abs_shap = np.abs(sv).mean(axis=0)   # should now be shape (n_features,) -- one scalar per feature
        assert mean_abs_shap.shape == (len(FEATURE_COLS),), f"Unexpected shape: {mean_abs_shap.shape}"

        shap_results[f"{hospital}__{class_name}"] = dict(zip(FEATURE_COLS, mean_abs_shap))
        print(f"  [{hospital}/{class_name}] n={len(pos_records)} — SHAP computed")

shap_summary_df = pd.DataFrame(shap_results).T
shap_summary_df.to_csv(r"C:\ECG_Project\training\All excel files\shap_feature_importance_summary.csv")
print(f"\nSaved: shap_feature_importance_summary.csv, shape={shap_summary_df.shape}")

  [chapman_shaoxing/AF] n=355 — SHAP computed
  [cpsc_2018/AF] n=275 — SHAP computed
  [georgia/AF] n=113 — SHAP computed
  [ningbo/AF] n=0, below min — skipped
  [ptb-xl/AF] n=303 — SHAP computed
  [chapman_shaoxing/IAVB] n=50 — SHAP computed
  [cpsc_2018/IAVB] n=166 — SHAP computed
  [georgia/IAVB] n=154 — SHAP computed
  [ningbo/IAVB] n=175 — SHAP computed
  [ptb-xl/IAVB] n=158 — SHAP computed
  [chapman_shaoxing/LAD] n=75 — SHAP computed
  [cpsc_2018/LAD] n=0, below min — skipped
  [georgia/LAD] n=186 — SHAP computed
  [ningbo/LAD] n=229 — SHAP computed
  [ptb-xl/LAD] n=1025 — SHAP computed
  [chapman_shaoxing/LBBB] n=38 — SHAP computed
  [cpsc_2018/LBBB] n=55 — SHAP computed
  [georgia/LBBB] n=44 — SHAP computed
  [ningbo/LBBB] n=49 — SHAP computed
  [ptb-xl/LBBB] n=106 — SHAP computed
  [chapman_shaoxing/NSIVCB] n=47 — SHAP computed
  [cpsc_2018/NSIVCB] n=0, below min — skipped
  [georgia/NSIVCB] n=41 — SHAP computed
  [ningbo/NSIVCB] n=107 — SHAP computed
  [ptb-xl/NSIVCB] n=157